In [ ]:
from typing import TypedDict, Literal, List, Dict, Any
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_core.language_models.chat_models import BaseChatModel

# ---------- State ----------
class GraphState(TypedDict, total=False):
    messages: List[dict]
    remaining_agents: List[str]
    collected_outputs: Dict[str, Any]   # {"schema": {...}, "attachment": {...}}
    next: str

# ---------- Router output schema ----------
class RouterDecision(TypedDict):
    next: str
    reason: str
    confidence: float

# ---------- Prompts ----------
SUPERVISOR_PROMPT = """
You are a supervisor router.
Pick ONE next node from available options.
When enough agent outputs are collected, choose "meta_analist".
Available options: {options}
"""

# ---------- Nodes ----------
def make_supervisor_node(llm: BaseChatModel):
    def supervisor_node(state: GraphState) -> Command[
        Literal[
            "schema",
            "attachment",
            "clinical_disorder",
            "cognetive_distortion",
            "functional_level",
            "personal_traits",
            "relational_pattern",
            "meta_analist",
            "__end__",
        ]
    ]:
        remaining = state.get("remaining_agents", [])
        # If nothing left, go synthesize
        if not remaining:
            return Command(goto="meta_analist", update={"next": "meta_analist"})

        options = remaining + ["meta_analist"]
        system_prompt = SUPERVISOR_PROMPT.format(options=", ".join(options))
        messages = [{"role": "system", "content": system_prompt}] + state["messages"]

        # structured output
        decision = llm.with_structured_output(RouterDecision).invoke(messages)
        nxt = decision["next"]

        # safety fallback
        if nxt not in options:
            nxt = "meta_analist"

        return Command(goto=nxt, update={"next": nxt})

    return supervisor_node


def make_agent_node(agent_name: str, agent_fn):
    def node(state: GraphState) -> Command[Literal["supervisor"]]:
        # agent_fn should return {"scores": {...}, "summary": "..."}
        result = agent_fn(state["messages"])

        outputs = dict(state.get("collected_outputs", {}))
        outputs[agent_name] = result

        remaining = [a for a in state.get("remaining_agents", []) if a != agent_name]

        return Command(
            goto="supervisor",
            update={
                "collected_outputs": outputs,
                "remaining_agents": remaining,
            },
        )
    return node


def make_meta_node(meta_fn):
    def meta_node(state: GraphState) -> Command[Literal["__end__"]]:
        # meta_fn gets all selected agent outputs and returns final synthesis JSON
        final_result = meta_fn(state.get("collected_outputs", {}))
        # Put into state/messages/store as needed
        return Command(goto=END, update={"final_result": final_result})
    return meta_node

# ---------- Build graph ----------
def build_graph(llm: BaseChatModel, agent_fns: Dict[str, Any], meta_fn):
    builder = StateGraph(GraphState)

    builder.add_node("supervisor", make_supervisor_node(llm))
    builder.add_node("meta_analist", make_meta_node(meta_fn))

    for name, fn in agent_fns.items():
        builder.add_node(name, make_agent_node(name, fn))
        builder.add_edge(name, "supervisor")  # agent -> supervisor loop

    builder.add_edge(START, "supervisor")
    builder.add_edge("meta_analist", END)

    return builder.compile()

# ---------- Example invocation ----------
# initial_state = {
#   "messages": [{"role": "user", "content": user_text}],
#   "remaining_agents": [
#       "schema", "attachment", "clinical_disorder",
#       "cognetive_distortion", "functional_level",
#       "personal_traits", "relational_pattern"
#   ],
#   "collected_outputs": {}
# }
# graph = build_graph(llm, agent_fns, meta_fn)
# result = graph.invoke(initial_state)